In [1]:
import numpy as np
import pandas as pd
import scipy
import re
import os
from pathlib import Path

Current indexing data downloaded from DOAJ (from https://doaj.org/, under Data > Journal CSV).
Past indexing data downloaded from https://docs.google.com/spreadsheets/d/183mRBRqs2jOyP0qZWXN8dUd02D4vL0Mov_kgYF8HORM/edit and https://docs.google.com/spreadsheets/d/1Kv3MbgFSgtSDnEGkA2JacrSjunRu0umHeZCtcMeqO5E/edit

In [5]:
current_df = pd.read_csv('../data/261028_doaj/doaj_journalcsv_20260927_2320_utf8.csv')

In [6]:
current_df['journal_title'] = current_df['Journal title']
current_df['internal_identifier'] = current_df['URL in DOAJ']
current_df['p_issn'] = current_df['Journal ISSN (print version)']
current_df['e_issn'] = current_df['Journal EISSN (online version)']
current_df['indexing_status'] = 'indexed'

In [7]:
past_df = pd.read_excel('../data/261028_doaj/Journals added to and withdrawn from DOAJ -  a change log.xlsx', 
                       sheet_name='Withdrawn', header=6).dropna(subset=['Journal Title'])
past_df['p_issn'] = past_df['ISSN']
past_df['Reason'] = past_df['Reason'].str.lower().str.strip()
reasons = past_df['Reason'] == 'journal not adhering to best practice'
past_df = past_df[reasons]
past_df['indexing_status'] = 'deindexed'
past_df['journal_title'] = past_df['Journal Title']

In [8]:
past_df = pd.read_excel('../data/261028_doaj/Journals added to and withdrawn from DOAJ -  a change log.xlsx', 
                       sheet_name='Withdrawn', header=6).dropna(subset=['Journal Title'])
past_df['p_issn'] = past_df['ISSN']
past_df['Reason'] = past_df['Reason'].str.lower().str.strip()
reasons = past_df['Reason'] == 'journal not adhering to best practice'
past_df = past_df[reasons]
past_df['indexing_status'] = 'deindexed'
past_df['journal_title'] = past_df['Journal Title']
#past_df['notes_on_indexing_status'] = past_df['Date Removed (dd/mm/yyyy)']
past_2024_df = past_df.copy()

In [9]:
past_df = pd.read_excel('../data/261028_doaj/DOAJ_ journals added and withdrawn [NO LONGER IN USE].xlsx', 
                       sheet_name='Withdrawn', header=6).dropna(subset=['Journal Title'])
past_df['p_issn'] = past_df['ISSN']
past_df['Reason'] = past_df['Reason'].str.lower().str.strip()
reasons = past_df['Reason'] == 'journal not adhering to best practice'
past_df = past_df[reasons]
past_df['indexing_status'] = 'deindexed'
past_df['journal_title'] = past_df['Journal Title']
#past_df['notes_on_indexing_status'] = past_df['Date Removed (dd/mm/yyyy)']
past_2014_df = past_df.copy()

In [10]:
df_all = pd.concat([current_df, past_2024_df, past_2014_df])
df_all['service'] = 'doaj'
df_all['notes_on_indexing_status'] = np.nan

In [11]:
df_all['edition'] = '261028'

In [12]:
df_all

,Journal title,Journal URL,URL in DOAJ,When did the journal start to publish all content using an open license?,Alternative title,Journal ISSN (print version),Journal EISSN (online version),Keywords,Languages in which the journal accepts manuscripts,Publisher,...,p_issn,e_issn,indexing_status,Journal Title,ISSN,Date Removed (dd/mm/yyyy),Reason,service,notes_on_indexing_status,edition
0,International Journal of Applied Science and E...,https://gigvvy.com/journals/ijase,https://doaj.org/toc/9b72dcc7886942629ba1cb261...,2020.0,NaN,1727-2394,1727-7841,"civil engineering, industrial engineering, app...",English,Chaoyang University of Technology,...,1727-2394,1727-7841,indexed,NaN,NaN,NaN,NaN,doaj,NaN,261028
1,مجله دامپزشکی ایران.,http://www.ivj.ir/?lang=en,https://doaj.org/toc/95d9a8f6e31a4cecabb49a525...,2012.0,Iranian Veterinary Journal,1735-6873,2676-704X,"veterinary, veterinary medicine","English, Persian",Shahid Chamran University of Ahvaz,...,1735-6873,2676-704X,indexed,NaN,NaN,NaN,NaN,doaj,NaN,261028
2,Krisis,https://krisis.eu/index,https://doaj.org/toc/be01a8448cce47cd8af6243e7...,2000.0,Journal for Contemporary Philosophy,NaN,1875-7103,"critical theory, political philosophy, social ...","Dutch, English",University of Groningen Press,...,NaN,1875-7103,indexed,NaN,NaN,NaN,NaN,doaj,NaN,261028
3,Harčova Nauka ì Tehnologìâ,http://fst.ontu.edu.ua/,https://doaj.org/toc/a0f006465ae045a5ac29989bb...,2014.0,Food Science and Technology,2073-8684,2409-7004,"food science, nutricology, food safety","Ukrainian, English",Odesa National University of Technology,...,2073-8684,2409-7004,indexed,NaN,NaN,NaN,NaN,doaj,NaN,261028
4,Akroterion,http://akroterion.journals.ac.za,https://doaj.org/toc/a0fd77cd21174c549e7059533...,1990.0,NaN,0303-1896,2079-2883,"classical studies, greek antiquity, roman anti...","Afrikaans, English","Stellenbsoch University, Department of Ancient...",...,0303-1896,2079-2883,indexed,NaN,NaN,NaN,NaN,doaj,NaN,261028
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5279,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1993-8233,NaN,deindexed,African Journal of Business Management,1993-8233,2016-09-06 00:00:00,journal not adhering to best practice,doaj,NaN,261028
5280,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1684-5315,NaN,deindexed,African Journal of Biotechnology,1684-5315,2016-09-06 00:00:00,journal not adhering to best practice,doaj,NaN,261028
5281,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1992-2248,NaN,deindexed,Scientific Research and Essays,1992-2248,2016-09-06 00:00:00,journal not adhering to best practice,doaj,NaN,261028
5282,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1538-2273,NaN,deindexed,Biotechnology and Molecular Biology Reviews,1538-2273,2016-09-06 00:00:00,journal not adhering to best practice,doaj,NaN,261028


In [13]:
df_all_slice = df_all[['service', 'edition', 'internal_identifier', 
                       'journal_title', 'p_issn', 'e_issn', 
                       'indexing_status', 'notes_on_indexing_status']].copy()

In [14]:
file_prefix = '261028_doaj'
df_all_slice.to_csv('../data/' + file_prefix + '.csv', index=False)
df_all_slice.to_parquet('../data/' + file_prefix + '.parquet')